## Part 0 — Is Ollama running?

This whole lab talks to **Ollama**, a local model server, over its REST API on port `11434`.

If the next cell doesn't print `Ollama is running`, open a terminal (PowerShell on Windows) and run `ollama serve`.
On Windows with the desktop app installed, just make sure the Ollama icon is in your system tray.

In [ ]:
import requests

try:
    r = requests.get("http://localhost:11434")
    print(r.text)            # should say: Ollama is running
except Exception as e:
    print("Could NOT reach Ollama:", e)
    print("Fix: open a terminal and run  ollama serve")

### Pull a small model zoo

We want **variety**: different families, different sizes, and one *distilled reasoner*.
Run the cell below (each pull is 1–5 GB — grab a coffee). If your machine is small, you can drop the bigger ones.

In [ ]:
# The ! prefix runs a shell command from inside the notebook.
!ollama pull llama3.2:3b
!ollama pull qwen2.5:3b
!ollama pull deepseek-r1:1.5b

In [ ]:
# What do we actually have locally? (names, sizes, digests)
!ollama list

## Part 1 — Two ways to call a model

Ollama gives you **two** HTTP APIs:

1. Its **native** API (`/api/generate`, `/api/chat`) — what the lab scripts use.
2. An **OpenAI-compatible** API (`/v1/...`) — lets you reuse the `openai` client library.

Let's see both. First the native generate endpoint:

In [ ]:
import time, requests

def generate(model, prompt):
    t0 = time.time()
    r = requests.post("http://localhost:11434/api/generate",
                      json={"model": model, "prompt": prompt, "stream": False,
                            "options": {"temperature": 0, "seed": 42}})
    r.raise_for_status()
    d = r.json()
    return d["response"].strip(), round(time.time()-t0, 2)

text, secs = generate("llama3.2:3b", "Tell me a fun fact in one sentence.")
print(f"({secs}s)  {text}")

Now the **Ollama** endpoint — same model, but called through the `openai` client.
Notice there's no OpenAI model involved; we just point the client at our local server.

In [ ]:
import ollama

resp = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "Tell me something interesting about octopuses."}],
)
print(resp.message.content)

## Part 2 — A tiny evaluation harness

Real model selection means testing every candidate on the **same fixed prompts** and recording
quality + latency. Here's a miniature version you can watch run.

The task suite is held constant — we never change a prompt to help one model (that would invalidate the comparison).

In [ ]:
TASKS = [
    {"id": "extract", "prompt": 'Return ONLY JSON with keys "city" and "date". '
        "Message: 'visiting the Toronto office on 2026-03-14.'"},
    {"id": "reason",  "prompt": "A train leaves 14:00, arrives 17:30, stopped 20 min. "
        "Minutes moving? Number only."},
    {"id": "instr",   "prompt": "List exactly three benefits of versioning data. "
        "Reply with exactly three lines starting with '- '. No preamble."},
]
MODELS = ["llama3.2:3b", "qwen2.5:3b", "deepseek-r1:1.5b"]
print(len(TASKS), "tasks x", len(MODELS), "models =", len(TASKS)*len(MODELS), "runs")

In [ ]:
rows = []
for m in MODELS:
    for t in TASKS:
        text, secs = generate(m, t["prompt"])
        rows.append({"model": m, "task": t["id"], "secs": secs, "text": text})
        print(f"[{m:<18}] {t['id']:<8} {secs:>5}s")
print("\ndone — collected", len(rows), "rows")

Let's look at the **same task across models** — this is where you start to *see* differences in quality.

In [ ]:
for r in rows:
    if r["task"] == "extract":
        print(f"--- {r['model']} ---")
        print(r["text"][:200], "\n")

## Part 3 — Prompt Lab: one model, seven prompt levers

This is the heart of the lab. We hold the **model fixed** and change only the **prompt**,
then watch the behavior change. Each "lever" changes exactly one thing.

We use the `/api/chat` endpoint because it supports a separate **system** prompt and a `format` option.

In [ ]:
import json, re

CHAT = "http://localhost:11434/api/chat"
PL_MODEL = "qwen2.5:3b"   # change to any tool-capable model you pulled

USER_MSG = "hey -- i'll swing by the TORONTO office on 03/14/2026, talk soon"
REASON_MSG = "A train leaves 14:00, arrives 17:30, stopped 20 min total. Minutes moving? Number only."

def chat(system, user, opts, fmt=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": user}]
    body = {"model": PL_MODEL, "messages": msgs, "stream": False, "options": opts}
    if fmt:
        body["format"] = fmt
    r = requests.post(CHAT, json=body, timeout=120); r.raise_for_status()
    return r.json()["message"]["content"].strip()

def is_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m: return False
    try: json.loads(m.group(0)); return True
    except Exception: return False

print("helpers ready, model =", PL_MODEL)

**Lever 1 — no system prompt.** The model free-forms. Expect prose, not clean data.

In [ ]:
out = chat(None, f"Extract the city and date: {USER_MSG}", {"temperature": 0})
print(out)
print("\nis_json:", is_json(out))

**Lever 2 — a role system prompt.** Telling it *what it is* ("a JSON API") changes the format with zero retraining.

In [ ]:
out = chat("You are a precise data-extraction API. Reply with JSON only, no prose.",
           f"Extract city and date: {USER_MSG}", {"temperature": 0})
print(out)
print("\nis_json:", is_json(out))

**Lever 3 — few-shot examples.** Showing 2 worked examples is often stronger than *instructing*.

In [ ]:
few = ("Examples:\n"
       "'meet in Paris on 2025-01-02' -> {\"city\":\"Paris\",\"date\":\"2025-01-02\"}\n"
       "'see you in Berlin 2025-05-09' -> {\"city\":\"Berlin\",\"date\":\"2025-05-09\"}\n"
       f"Now: '{USER_MSG}' ->")
out = chat("You extract fields as JSON.", few, {"temperature": 0})
print(out)
print("\nis_json:", is_json(out))

**Lever 4 — forced JSON** (`format="json"`). Ollama constrains decoding so the output is *guaranteed* parseable.

In [ ]:
out = chat("Extract city and date.", USER_MSG, {"temperature": 0}, fmt="json")
print(out)
print("\nis_json:", is_json(out))

**Lever 5 — temperature.** Same prompt, run 3x at high temperature. Watch it produce **different** outputs each time — this is why reproducible tests pin temperature and seed.

In [ ]:
print("=== temperature 1.2 (non-deterministic) ===")
outs = [chat("Extract the city and date in one sentence.", USER_MSG, {"temperature": 1.2})
        for _ in range(3)]
for i, o in enumerate(outs): print(f"run {i}: {o}")
print("distinct outputs:", len({*outs}))

print("\n=== temperature 0 + fixed seed (reproducible) ===")
outs0 = [chat("Extract the city and date in one sentence.", USER_MSG,
              {"temperature": 0, "seed": 42}) for _ in range(3)]
for i, o in enumerate(outs0): print(f"run {i}: {o}")
print("distinct outputs:", len({*outs0}))

**Lever 6 — persona / tone.** Same facts, but now a warm support agent. The *same model* serves an API or a chatbot purely via the prompt.

In [ ]:
out = chat("You are a cheerful support agent. Confirm the visit warmly in 2 sentences.",
           USER_MSG, {"temperature": 0.4})
print(out)

**Lever 7 — chain-of-thought.** On a reasoning task, asking it to *think step by step* trades tokens for accuracy.

In [ ]:
out = chat("Think step by step, then give the final number on its own last line.",
           REASON_MSG, {"temperature": 0})
print(out)

### Your turn (exercise)

Add an **eighth lever** of your own below — for example a *negative* instruction
(`"Never include any explanation — output only the JSON object."`). **Predict** the behavior first,
then run it and check.

In [ ]:
# TODO: write your own system prompt and run chat(...) with it.
my_system = "Never include any explanation -- output only the JSON object."
out = chat(my_system, USER_MSG, {"temperature": 0})
print(out)
print("\nis_json:", is_json(out))

## Part 4 — Score & select on priorities

A scorecard turns raw outputs into a decision. We auto-check each task, then show that
**different priorities pick different winners** — the core MLOps lesson: *"best model" is undefined without priorities.*

In [ ]:
def score_extract(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m: return 0.0
    try:
        o = json.loads(m.group(0))
        return 1.0 if str(o.get("city","")).lower()=="toronto" else 0.5
    except Exception:
        return 0.0

def score_reason(text):
    return 1.0 if "190" in text else 0.0

# score the extract + reason rows we collected in Part 2
from statistics import mean
by_model = {}
for r in rows:
    if r["task"] == "extract": s = score_extract(r["text"])
    elif r["task"] == "reason": s = score_reason(r["text"])
    else: continue
    by_model.setdefault(r["model"], []).append(s)

print(f"{'model':<20}{'quality':>8}")
for m, ss in by_model.items():
    print(f"{m:<20}{mean(ss):>8.2f}")

Now combine quality with latency under two personas. Edit the weights and watch the winner change.

In [ ]:
# average latency per model from Part 2
lat = {}
for r in rows:
    lat.setdefault(r["model"], []).append(r["secs"])
lat = {m: mean(v) for m, v in lat.items()}

def normalize(d):  # min-max to [0,1]
    lo, hi = min(d.values()), max(d.values())
    return {k: (0.0 if hi==lo else (v-lo)/(hi-lo)) for k, v in d.items()}

q = normalize({m: mean(ss) for m, ss in by_model.items()})
l = normalize(lat)

def rank(w_q, w_l, label):
    print(f"\n{label}  (w_q={w_q}, w_l={w_l})")
    scored = {m: w_q*q[m] - w_l*l[m] for m in q}
    for m, s in sorted(scored.items(), key=lambda x: -x[1]):
        print(f"  {m:<20}{s:>7.2f}")

rank(0.3, 0.7, "Persona A: interactive (latency matters)")
rank(0.9, 0.0, "Persona B: overnight batch (quality first)")

## Part 5 — Agentware: a ReAct tool-calling loop

Now the model stops just *returning text* and starts *taking actions*. We give it two tools and let it
decide when to call them, in a **reason → act → observe** loop. Use a tool-capable model (`qwen2.5`, `llama3.1`).

In [ ]:
import ollama

ORDERS = {"A100": {"status": "shipped", "total": 240.0},
          "B200": {"status": "processing", "total": 80.0}}

def lookup_order(order_id: str) -> str:
    """Return the status and total of a customer order."""
    o = ORDERS.get(order_id)
    return json.dumps(o) if o else "unknown"

def calculate(expr: str) -> str:
    """Evaluate a simple arithmetic expression like '240 * 0.05'."""
    if not all(c in "0123456789.+-*/() " for c in expr):
        return "error: arithmetic only"
    return str(eval(expr, {"__builtins__": {}}, {}))

TOOLS = {"lookup_order": lookup_order, "calculate": calculate}
AGENT_MODEL = "qwen2.5:3b"
print("tools:", list(TOOLS))

The task is **two-hop**: it must look up the order *first*, then use that total in a calculation.
A single call can't do it — that's why we loop.

In [ ]:
def run_agent(user_msg, max_steps=6):
    msgs = [{"role": "system",
             "content": "You are an orders assistant. Always look up the order before "
                        "answering. Use calculate for any arithmetic; never do math yourself."},
            {"role": "user", "content": user_msg}]
    for step in range(max_steps):
        resp = ollama.chat(model=AGENT_MODEL, messages=msgs, tools=list(TOOLS.values()))
        m = resp.message
        msgs.append(m)
        if not m.tool_calls:
            print("FINAL:", m.content); return
        for tc in m.tool_calls:
            name, args = tc.function.name, tc.function.arguments
            result = TOOLS[name](**args)
            print(f"  ACT step {step}: {name}({args}) -> {result}")
            msgs.append({"role": "tool", "name": name, "content": result})

run_agent("If order A100 has shipped, what is the 5% loyalty credit on its total?")

## Part 6 — MCP: standardizing tools so they're swappable

In Part 5 the tools were wired into the agent by hand. **MCP (Model Context Protocol)** moves them behind
a *server* the agent can **discover** at runtime. The big win: add a tool to the server and the client
finds it automatically — no agent code changes.

The starter repo has `mcp_server.py` and `mcp_client.py`. Run the client from a terminal:

```bash
python mcp_client.py
```

It spawns the server over stdio and walks the lecture's 5 steps:

1. **discover** the server's tools
2. **add** their descriptions to the model prompt
3. **intercept** the model's tool request
4. **invoke** the tool through MCP
5. **return** the result to the model

Below is the discovery step on its own, so you can see step 1 from inside the notebook.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def discover():
    params = StdioServerParameters(command="python", args=["mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()              # STEP 1: discover
            print("discovered tools:", [t.name for t in tools.tools])
            result = await session.call_tool("lookup_order", {"order_id": "A100"})  # STEP 4
            print("invoke lookup_order(A100) ->", result.content[0].text)

# In Jupyter use await directly; in a plain script use asyncio.run(discover())
await discover()

**Swappability exercise:** open `mcp_server.py`, add a third `@mcp.tool()` (e.g. `order_total`),
re-run the cell above, and confirm the discovered-tools list grows from 2 to 3 — **without editing this notebook**.
That is the entire point of MCP.

## Wrap-up & deliverables

You've now touched the full arc: **model → prompt → selection → tools → protocol**. For the full Lab 6
write-up (data pipeline, CI quality gate, monitoring/flywheel, AI-SBOM, and the 7-principles audit),
continue in the Lab 6 manual. Submit:

- `ollama list` output + which model is distilled and from what
- the prompt-lab observations (which lever changed behavior most; your 3→1 determinism result; your 8th lever)
- the two persona rankings (and whether they pick different winners)
- the agent's two-hop transcript
- the MCP discovered-tools list before and after adding a tool

**Tip:** if a cell is slow, switch `PL_MODEL` / `AGENT_MODEL` to `llama3.2:3b` or a `:1b` variant.